<a href="https://colab.research.google.com/github/aszczi/Urban_mobility_in_Cracow/blob/main/Opoznienia_KMK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analiza Opóźnień Komunikacji Miejskiej w Krakowie (GTFS-RT)

Niniejszy notatnik służy do analizy rzeczywistych opóźnień komunikacji miejskiej w Krakowie. Dane pobierane są w czasie rzeczywistym z usług [GTFS-RT ZTP Kraków](https://gtfs.ztp.krakow.pl/). 

Wykorzystujemy:
- **TripUpdates** (format `.pb` - Protobuf), aby pozyskać estymowane czasy przyjazdów i porównać je do planowanych.
- **Dane statyczne (GTFS)** do podpięcia lokalizacji geo (przystanków) oraz nazw linii.

Notatnik wygeneruje interaktywne mapy ulic ukazujące natężenie opóźnień, a także odpowiednie statystyki i wykresy.

In [ ]:
# Jeśli nie masz ich zainstalowanych, odkomentuj i uruchom poniższą linijkę:
!pip install gtfs-realtime-bindings protobuf requests pandas plotly folium scipy osmnx networkx matplotlib
import requests
from google.transit import gtfs_realtime_pb2
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import TimestampedGeoJson
import osmnx as ox
import networkx as nx
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import datetime
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def fetch_gtfs_rt_delays():
    print("Pobieranie aktualnych danych GTFS-RT (TripUpdates) dla Krakowa...")

    urls = {
        "Tramwaje": "https://gtfs.ztp.krakow.pl/TripUpdates_T.pb",
        "Autobusy": "https://gtfs.ztp.krakow.pl/TripUpdates_A.pb"
    }

    delays_data = []

    for v_type, url in urls.items():
        print(f" Pobieranie danych dla: {v_type}...")
        try:
            feed = gtfs_realtime_pb2.FeedMessage()
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            feed.ParseFromString(response.content)
            
            for entity in feed.entity:
                if entity.HasField('trip_update'):
                    trip_id = entity.trip_update.trip.trip_id
                    route_id = entity.trip_update.trip.route_id
                    
                    for stu in entity.trip_update.stop_time_update:
                        stop_id = stu.stop_id
                        delay = None
                        
                        # Pobieranie opóźnienia z przyjazdu lub odjazdu (preferujemy przyjazd)
                        if stu.HasField('arrival') and stu.arrival.HasField('delay'):
                            delay = stu.arrival.delay
                        elif stu.HasField('departure') and stu.departure.HasField('delay'):
                            delay = stu.departure.delay
                        
                        if delay is not None:
                            arr_time = None
                            if stu.HasField('arrival') and stu.arrival.HasField('time'):
                                arr_time = stu.arrival.time
                            elif stu.HasField('departure') and stu.departure.HasField('time'):
                                arr_time = stu.departure.time
                            
                            delays_data.append({
                                "typ": v_type,
                                "trip_id": trip_id,
                                "line_num": route_id,
                                "stop_sequence": stu.stop_sequence if stu.HasField('stop_sequence') else 0,
                                "stop_id": stop_id,
                                "delay_sec": delay,
                                "delay_min": delay / 60.0,
                                "time": arr_time
                            })
        except Exception as e:
            print(f"  Błąd podczas pobierania {v_type}: {e}")

    df_delays = pd.DataFrame(delays_data)
    
    if not df_delays.empty:
        # Posiadamy delay_sec (opóźnienia dodatnie oznaczają spóźnienie, pomijamy te < 0, bo to znaczy przyspieszenie)
        df_delays = df_delays[df_delays['delay_sec'] > 0]
        
        # Konwersja czasu Uniksowego do datetime i godziny ISO
        if 'time' in df_delays.columns:
            df_delays['datetime'] = pd.to_datetime(df_delays['time'], unit='s')
            df_delays['hour'] = df_delays['datetime'].dt.strftime('%Y-%m-%dT%H:00:00')
            # Jeżeli null (brak time) to przypisujemy bieżącą
            df_delays['hour'] = df_delays['hour'].fillna(datetime.datetime.now().strftime('%Y-%m-%dT%H:00:00'))
        else:
            df_delays['hour'] = datetime.datetime.now().strftime('%Y-%m-%dT%H:00:00')
            
        print(f"\nZakończono. Pobrano {len(df_delays)} rekordów z dodatnimi opóźnieniami.")
    else:
        print("\nNie udało się pobrać żadnych opóźnień lub brak opóźnień w tej chwili.")
        
    return df_delays

df_delays = fetch_gtfs_rt_delays()
df_delays.head()

In [ ]:
# Wczytanie fizycznych lokalizacji przystanków i nazw linii z rozkładów (GTFS Zip)
# Zakładamy, że historyczne (ale w miarę aktualne) pliki przystanków znajdują się lokalnie
static_gtfs_dir = "data/GTFS_ZTP_17.05.26/"
stops_file = os.path.join(static_gtfs_dir, "stops.txt")
routes_file = os.path.join(static_gtfs_dir, "routes.txt")

if os.path.exists(stops_file) and not df_delays.empty:
    df_stops = pd.read_csv(stops_file, dtype=str)
    # Konwersja coords na wartości numeryczne
    df_stops["stop_lat"] = pd.to_numeric(df_stops["stop_lat"], errors="coerce")
    df_stops["stop_lon"] = pd.to_numeric(df_stops["stop_lon"], errors="coerce")
    
    # Łączenie przystanków
    df_merged = df_delays.merge(
        df_stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], 
        on='stop_id', 
        how='inner'
    )
    
    # Przypisywanie nazw linii, jeżeli istnieje routes.txt
    if os.path.exists(routes_file):
        df_routes = pd.read_csv(routes_file, dtype=str)
        df_merged = df_merged.merge(
            df_routes[['route_id', 'route_short_name']], 
            left_on='line_num', 
            right_on='route_id',
            how='left'
        )
        # Zastąpienie wewnętrznego ID linii jej nazwą publiczną np. "152"
        df_merged['linia'] = df_merged['route_short_name'].fillna(df_merged['line_num'])
    else:
        df_merged['linia'] = df_merged['line_num']

    # Obliczanie średniego opóźnienia, maksymalnego, oraz ilości pojazdów per Przystanek
    df_stops_delays = df_merged.groupby(['stop_name', 'stop_lat', 'stop_lon', 'typ'], as_index=False).agg(
        mean_delay_min=('delay_min', 'mean'),
        max_delay_min=('delay_min', 'max'),
        measurements_count=('delay_min', 'count')
    )
    
    # Wyświetlamy tylko te przystanki, przez które opóźnione przejeżdża więcej niż x pojazdów
    df_stops_delays = df_stops_delays[df_stops_delays['measurements_count'] >= 2]
    
    display(df_stops_delays.sort_values(by='mean_delay_min', ascending=False).head())
else:
    print("Brak pliku przystanków lub brak punktów pobranych - upewnij się, ze ścieżka do stops.txt jest poprawna.")

## Dynamiczna mapa ruchu komunikacji miejskiej
Nakładamy dane z przystanków na mapę i animujemy je z użyciem GeoJSON. Używamy OSMnx do znalezienia najbliższych dróg.

In [ ]:
from IPython.display import display

if 'df_merged' in locals() and not df_merged.empty:
    
    # 1. Pobranie grafu drogowego miasta z OSMnx
    lokalizacja = "Kraków, Poland"
    print(f"Pobieranie geometrii dróg dla: {lokalizacja}... ")
    G = ox.graph_from_place(lokalizacja, network_type="drive", simplify=True)
    
    unique_all_points = df_merged.drop_duplicates(subset=['stop_name', 'stop_lat', 'stop_lon']).copy()
    
    print("Przypinanie przystanków do siatki ulic Krakowa...")
    lats = unique_all_points['stop_lat'].values
    lons = unique_all_points['stop_lon'].values
    
    try:
        nearest_edges = ox.nearest_edges(G, X=lons, Y=lats)
    except AttributeError:
        nearest_edges = ox.distance.nearest_edges(G, X=lons, Y=lats)
        
    point_to_edge = dict(zip(unique_all_points['stop_name'], nearest_edges))
    
    features = []
    cmap = plt.get_cmap('RdYlGn_r') 
    norm = mcolors.Normalize(vmin=0, vmax=30)
    
    unique_hours = sorted(df_merged['hour'].unique())
    print("Przetwarzanie opóźnień w przedziały godzinowe dla Folium...")
    
    for h in unique_hours:
        df_hour = df_merged[df_merged['hour'] == h]
        
        for idx, row in df_hour.iterrows():
            p_name = row['stop_name']
            delay = row['delay_min']
            
            if p_name not in point_to_edge:
                continue
                
            edge = point_to_edge[p_name]
            u, v, key = edge
            
            coords = [[G.nodes[u]['x'], G.nodes[u]['y']], [G.nodes[v]['x'], G.nodes[v]['y']]]
            
            features.append({
                "type": "Feature",
                "geometry": {
                    "type": "LineString",
                    "coordinates": coords
                },
                "properties": {
                    "times": [h] * 2, 
                    "style": {
                        "color": mcolors.to_hex(cmap(norm(delay))),
                        "weight": 8,
                        "opacity": 0.85
                    }
                }
            })
            
    print("Tworzenie docelowej Interaktywnej Mapy z kolorowaniem ulic z całego miasta...")
    folium_map = folium.Map(location=[50.0614, 19.9383], zoom_start=13, tiles="cartodbdark_matter")
    
    TimestampedGeoJson(
        {"type": "FeatureCollection", "features": features},
        period="PT1H",
        add_last_point=False,
        auto_play=True,
        loop=True,
        max_speed=1,
        loop_button=True,
        time_slider_drag_update=True
    ).add_to(folium_map)
    
    display(folium_map)
else:
    print("Brak danych (df_merged) do wyświetlenia mapy.")

## Wykresy (Gdzie są największe opóźnienia)
Przeanalizujmy, które przystanki oraz które linie notują średnio największe opóźnienia w pozyskanej próbce czasowej.

In [ ]:
if 'df_stops_delays' in locals() and not df_stops_delays.empty:
    # Top 15 Przystanków
    top_15_mean = df_stops_delays.sort_values(by='mean_delay_min', ascending=False).head(15)

    fig_bar_stops = px.bar(
        top_15_mean,
        x='stop_name',
        y='mean_delay_min',
        color='typ',
        title="Top 15 przystanków o największym średnim opóźnieniu",
        labels={'stop_name': 'Przystanek', 'mean_delay_min': 'Średnie opóźnienie (min)'},
        text_auto=':.1f',
        height=500
    )
    fig_bar_stops.update_layout(xaxis_tickangle=-45)
    fig_bar_stops.show()
    
if 'df_merged' in locals() and not df_merged.empty:
    # Agregacja po Liniach
    df_route_delays = df_merged.groupby(['linia', 'typ'], as_index=False).agg(
        mean_delay_min=('delay_min', 'mean'),
        measurements_count=('delay_min', 'count')
    )
    
    # Filtrujemy by odrzucić pojedyncze strzały pomiarów dla jakiejś trasy
    df_route_delays = df_route_delays[df_route_delays['measurements_count'] >= 3]
    
    top_15_routes = df_route_delays.sort_values(by='mean_delay_min', ascending=False).head(15)
    
    fig_bar_routes = px.bar(
        top_15_routes,
        x='linia',
        y='mean_delay_min',
        color='typ',
        title="Jakie linie najwięcej się spóźniają? (Top 15)",
        labels={'linia': 'Numer Linii', 'mean_delay_min': 'Średnie opóźnienie (min)'},
        text_auto=':.1f',
        height=500
    )
    fig_bar_routes.update_layout(xaxis_type='category') # By numery linii zachowywały się jak kategorie
    fig_bar_routes.show()

    # Wykres godzinowy
    df_hourly = df_merged.groupby('hour', as_index=False).agg(
        avg_delay_min=('delay_min', 'mean')
    ).sort_values(by='hour')

    fig_time = px.line(
        df_hourly, 
        x='hour', 
        y='avg_delay_min', 
        markers=True,
        title="W jakich godzinach jest najwięcej spóźnień KMK?",
        labels={'hour': 'Godzina', 'avg_delay_min': 'Średnie opóźnienie (min)'}
    )
    fig_time.show()
